In [5]:
import os
import json
import sys
import argparse
import datetime
import pandas as pd
from trade_core import InitCore
import time
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [2]:
redis_cli = InitRedis()

In [ ]:
temp = pickle.loads(redis_cli.get_data('INFO_CORE|UPBIT_SPOT/KRW:BITHUMB_SPOT/KRW_1T_now'))
tempexit

In [6]:
logging_dir = "/home/trade_core/"
with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)
proc_n = 3
# node = config['node']
node = 'trade_core1'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
master_flag = config['node_settings'][node]['MASTER']
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets = config['node_settings'][node]['enabled_markets']

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [7]:
core = InitCore(logging_dir, master_flag, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, enabled_markets, db_dict)


2023-11-28 18:25:56,543 - info_core - INFO - InitCore|InitCore initiated with proc_n=3


2023-11-28 18:25:56,658 - okx_plug - INFO - okx_plug_logger started.
2023-11-28 18:25:56,924 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-28 18:25:57,049 - binance_plug - INFO - binance_plug_logger started.
2023-11-28 18:25:57,050 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-11-28 18:25:57,052 - bybit_plug - INFO - bybit_plug_logger started.
2023-11-28 18:25:57,054 - info_core - INFO - InitCore|update_bybit_usd_m_info_df thread has started.
2023-11-28 18:25:57,055 - info_core - INFO - InitCore|update_bybit_usd_m_ticker_df thread has started.
2023-11-28 18:25:57,057 - info_core - INFO - InitCore|update_okx_usd_m_ticker_df thread has started.
2023-11-28 18:25:57,058 - info_core - INFO - InitCore|update_okx_usd_m_info_df thread has started.
2023-11-28 18:25:57,058 - info_core - INFO - InitCore|update_binance_spot_ticker_df thread has started.
2023-11-28 18:25:57,060 - info_core - INFO - InitCore|update_binance_spot_info_df thread has started.
2023-11-28 18:25:57,0

2023-11-28 18:28:02,025 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:28:32,882 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:29:03,625 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:29:34,498 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:30:05,379 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:30:36,240 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:31:07,102 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.
2023-11-28 18:31:37,952 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.


In [28]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
start = time.time()
for i in range(30):
    temp = core.get_premium_df(target_market_code, origin_market_code)
time.time() - start

2.3683786392211914

2023-11-28 18:32:39,656 - update_dollar - INFO - fetch_dollar|Dollar price (1296.0 KRW) has been updated.


In [4]:
test_dict = {'a': 1}


def get_test_dict():
    temp = lambda :test_dict
    return temp()


get_test_dict()

{'a': 1}

In [ ]:
core.enabled_markets_dict

In [ ]:
target_market = 'UPBIT_SPOT'
# def get_symbol_list(core, target_market): # E.g) UPBIT_SPOT, BINANCE_SPOT, BINANCE_USD_M, BINANCE_COIN_M

target_exchange = target_market.split('_')[0]
target_market_type = '_'.join(target_market.split('_')[1:])

comparing_exchanges = core.enabled_markets_dict.keys()
comparison_list = []
total_df = pd.DataFrame()
for exchange in comparing_exchanges:
    for market_type in core.enabled_markets_dict[exchange]:
        if market_type == "SPOT":
            for quote_asset in core.enabled_markets_dict[exchange][market_type]:
                comparison_list.append({"exchange": exchange, "market_type": market_type, "contract_type":None, "quote_asset": quote_asset})
                # TEST
                # print({"exchange": exchange, "market_type": market_type, "contract_type":None, "quote_asset": quote_asset})
        else:
            for contract_type in core.enabled_markets_dict[exchange][market_type]:
                for quote_asset in core.enabled_markets_dict[exchange][market_type][contract_type]:
                    comparison_list.append({"exchange": exchange, "market_type": market_type, "contract_type":contract_type, "quote_asset": quote_asset})

for i, comparison_dict in enumerate(comparison_list):
    if comparison_dict['exchange'] == target_exchange and comparison_dict['market_type'] == target_market_type:
        target_market_dict = comparison_list.pop(i)
        break

# Start compare and concat
target_market_symbols = []
target_market_ticker_df = core.info_dict[f"{target_market_dict['exchange'].lower()}_{target_market_dict['market_type'].lower()}_ticker_df"]
# check if it's spot or not
if target_market_dict['market_type'] != "SPOT":
    target_market_info_df = core.info_dict[f"{target_market_dict['exchange'].lower()}_{target_market_dict['market_type'].lower()}_info_df"][['symbol','perpetual']]
    target_market_ticker_df = target_market_ticker_df.merge(target_market_info_df, on='symbol', how='inner')
    if target_market_dict['contract_type'] == "PERPETUAL":
        target_market_ticker_df = target_market_ticker_df[target_market_ticker_df['perpetual'] == True]
    else: # FUTURES
        target_market_ticker_df = target_market_ticker_df[target_market_ticker_df['perpetual'] == False]
target_market_ticker_df = target_market_ticker_df[target_market_ticker_df['quote_asset']==target_market_dict['quote_asset']][['symbol','lastPrice','atp24h','base_asset','quote_asset']]
for each_comparison_dict in comparison_list:
    each_market_info_df = core.info_dict[f"{each_comparison_dict['exchange'].lower()}_{each_comparison_dict['market_type'].lower()}_info_df"]
    if each_comparison_dict['contract_type'] is None:
        each_market_info_df = each_market_info_df[each_market_info_df['quote_asset']==each_comparison_dict['quote_asset']]
    else: # contract_type is PERPETUAL or FUTURES
        if each_comparison_dict['contract_type'] == "PERPETUAL":
            each_market_info_df = each_market_info_df[(each_market_info_df['quote_asset']==each_comparison_dict['quote_asset'])&(each_market_info_df['perpetual'] == True)]
        else: # FUTURES
            each_market_info_df = each_market_info_df[(each_market_info_df['quote_asset']==each_comparison_dict['quote_asset'])&(each_market_info_df['perpetual'] == False)]
    comparison_market_code = f"{each_comparison_dict['exchange'].lower()}_{each_comparison_dict['market_type'].lower()}"
    new_symbol = f"symbol_{comparison_market_code}"
    new_quote_asset = f"quote_asset_{comparison_market_code}"
    each_market_info_df = each_market_info_df.rename(columns={"symbol":new_symbol, "quote_asset": new_quote_asset})

    merged_df = target_market_ticker_df.merge(each_market_info_df[[new_symbol, "base_asset", new_quote_asset]], on='base_asset', how='inner')
    if (each_comparison_dict['exchange'] == target_market_dict['exchange'] and
        each_comparison_dict['market_type'] == target_market_dict['market_type']):
        target_market_symbols += merged_df[new_symbol].to_list()
    total_df = pd.concat([total_df, merged_df], axis=0, ignore_index=True)
    target_market_symbols += total_df['symbol'].to_list()

target_market_symbols = list(set(target_market_symbols))
total_target_market_ticker_df = core.info_dict[f"{target_market_dict['exchange'].lower()}_{target_market_dict['market_type'].lower()}_ticker_df"]
total_target_market_df = total_target_market_ticker_df[total_target_market_ticker_df['symbol'].isin(target_market_symbols)]
final_symbol_list = total_target_market_df.sort_values('atp24h', ascending=False)['symbol'].to_list()

# total_df.drop_duplicates(['symbol'], inplace=True)
# total_df.sort_values('atp24h', ascending=False)
# total_df.reset_index(drop=True, inplace=True)
# return total_df['symbol'].to_list()
# get_symbol_list(core, 'UPBIT_SPOT')

In [ ]:
len(final_symbol_list)

In [ ]:
target_market_symbols

In [ ]:
len(final_symbol_list)

In [ ]:
same_market_symbols

In [ ]:
total_df

In [ ]:
temp = core.get_symbol_list("BINANCE_SPOT")
temp

In [ ]:
[x for x in temp if 'BTC' in x]

In [ ]:
core.get_premium_df('UPBIT_SPOT/BTC', "BINANCE_USD_M/USDT")

In [ ]:
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"
core.get_premium_df(target_market_code, origin_market_code)

In [ ]:
origin_market = "UPBIT_SPOT"
target_market = "BINANCE_SPOT"

origin_market_spot_info_df = core.info_dict['binance_spot_info_df']
origin_quote_asset = 'BTC'
target_quote_asset = 'USDT'

if origin_quote_asset == "USD":
    origin_quote_asset = "USDT"
if target_quote_asset == "USD":
    target_quote_asset = "USDT"
if origin_quote_asset == target_quote_asset:
    # return 1
    print(1) # TEST
# origin_market_spot_info_df = self.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
origin_market_spot_info_df = core.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"] # TEST
# First try to find the rate from the info_dict

def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
    df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
    reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
    if len(df) == 1:
        convert_rate = df['lastPrice'].values[0]
    elif len(reverse_df) == 1:
        convert_rate = 1 / reverse_df['lastPrice'].values[0]
    else:
        convert_rate = None
    return convert_rate
convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
if convert_rate is None: # not between coins
    # print("1st convert_rate is None, Not between coins")
    if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
        # convert_rate = self.get_dollar_dict()['price']
        convert_rate = core.get_dollar_dict()['price'] # TEST
    elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
        # convert_rate = 1 / self.get_dollar_dict()['price']
        convert_rate = 1 / core.get_dollar_dict()['price'] # TEST
    elif target_quote_asset == "KRW":
        # convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * self.get_dollar_dict()['price']
        convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * core.get_dollar_dict()['price'] # TEST
    elif origin_quote_asset == "KRW":
        # temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
        temp_convert_rate = core.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset) # TEST
        if temp_convert_rate is not None:
            convert_rate = 1 / temp_convert_rate
        else:
            title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    else:
        # temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, "USDT")
        temp_convert_rate = core.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset) # TEST
        if temp_convert_rate is not None:
            convert_rate = 1 / temp_convert_rate
            # return convert_rate
            print(convert_rate)
        else:
            pass
        title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
        raise Exception(f"Cannot find the convert rate for {title}")
# return convert_rate
print(convert_rate) # TEST

In [ ]:
origin_market_spot_info_df.columns

In [ ]:
origin_market_spot_info_df[['market','base_asset','quote_asset']]#[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]

In [ ]:
temp_convert_rate

In [ ]:
def convert_asset_rate(self, origin_market, origin_quote_asset, target_market, target_quote_asset):
    if origin_quote_asset == "USD":
        origin_quote_asset = "USDT"
    if target_quote_asset == "USD":
        target_quote_asset = "USDT"
    if origin_quote_asset == target_quote_asset:
        return 1
    origin_market_spot_info_df = self.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
    # First try to find the rate from the info_dict

    def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
        df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
        reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
        if len(df) == 1:
            convert_rate = df['lastPrice'].values[0]
        elif len(reverse_df) == 1:
            convert_rate = 1 / reverse_df['lastPrice'].values[0]
        else:
            convert_rate = None
        return convert_rate
    convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
    if convert_rate is None: # not between coins
        # print("1st convert_rate is None, Not between coins")
        if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
            convert_rate = self.get_dollar_dict()['price']
        elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
            convert_rate = 1 / self.get_dollar_dict()['price']
        elif target_quote_asset == "KRW":
            convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * self.get_dollar_dict()['price']
        elif origin_quote_asset == "KRW":
            temp_convert_rate = self.convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
            if temp_convert_rate is not None:
                convert_rate = 1 / temp_convert_rate
            else:
                title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
                raise Exception(f"Cannot find the convert rate for {title}")
        else:
            title = f"target_market: {target_market}, target_quote_asset: {target_quote_asset}, origin_market:{origin_market}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    return convert_rate

In [ ]:
core.get_market_code_list()

In [ ]:
origin_exchange_code = 'BINANCE_SPOT/BTC'
target_exchange_code = 'UPBIT_SPOT/KRW'

core.get_premium_df(origin_exchange_code, target_exchange_code)

In [ ]:
core.exchange_websocket_dict['UPBIT_SPOT'].check_status(print_result=True)

In [ ]:
core.exchange_websocket_dict['BINANCE_SPOT'].check_status(print_result=True)

In [ ]:
core.exchange_websocket_dict['BINANCE_USD_M'].get_price_df()

In [ ]:
temp = core.exchange_websocket_dict['BINANCE_USD_M'].get_price_df()
temp[temp['base_asset']=='STX']

In [ ]:
temp = core.exchange_websocket_dict['UPBIT_SPOT'].get_price_df()
temp[temp['base_asset']=='STX']

In [ ]:
core.info_dict['binance_spot_ticker_df']['symbol']

In [ ]:
len(core.get_binance_spot_symbol_list())

In [ ]:
core.info_dict['upbit_spot_ticker_df']

In [ ]:
def convert_asset_rate(origin_market, origin_quote_asset, target_market, target_quote_asset):
    if origin_quote_asset == "USD":
        origin_quote_asset = "USDT"
    if target_quote_asset == "USD":
        target_quote_asset = "USDT"
    if origin_quote_asset == target_quote_asset:
        return 1
    origin_market_spot_info_df = core.info_dict[f"{origin_market.lower().split('_')[0]}_spot_ticker_df"]
    # First try to find the rate from the info_dict

    def convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset):
        df = origin_market_spot_info_df[(origin_market_spot_info_df['base_asset']==origin_quote_asset)&(origin_market_spot_info_df['quote_asset']==target_quote_asset)]
        reverse_df = origin_market_spot_info_df[(origin_market_spot_info_df['quote_asset']==origin_quote_asset)&(origin_market_spot_info_df['base_asset']==target_quote_asset)]
        if len(df) == 1:
            convert_rate = df['lastPrice'].values[0]
        elif len(reverse_df) == 1:
            convert_rate = 1 / reverse_df['lastPrice'].values[0]
        else:
            convert_rate = None
        return convert_rate
    convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, target_quote_asset)
    if convert_rate is None: # not between coins
        # print("1st convert_rate is None, Not between coins")
        if target_quote_asset == "KRW" and origin_quote_asset == "USDT":
            convert_rate = core.update_dollar_return_dict['price']
        elif target_quote_asset == "USDT" and origin_quote_asset == "KRW":
            convert_rate = 1 / core.update_dollar_return_dict['price']
        elif target_quote_asset == "KRW":
            convert_rate = convert_between_coins(origin_market_spot_info_df, origin_quote_asset, "USDT") * core.update_dollar_return_dict['price']
        elif origin_quote_asset == "KRW":
            temp_convert_rate = convert_asset_rate(target_market, target_quote_asset, origin_market, origin_quote_asset)
            if temp_convert_rate is not None:
                convert_rate = 1 / temp_convert_rate
            else:
                title = f"target_quote_asset: {target_quote_asset}, origin_quote_asset: {origin_quote_asset}"
                raise Exception(f"Cannot find the convert rate for {title}")
        else:
            title = f"target_quote_asset: {target_quote_asset}, origin_quote_asset: {origin_quote_asset}"
            raise Exception(f"Cannot find the convert rate for {title}")
    return convert_rate

In [ ]:
exchange_origin = 'BINANCE_USD_M/USDT'
exchange_target = 'UPBIT_SPOT/BTC'
def get_premium_df(self, target_exchange_code, origin_exchange_code):
    # POSSIBLE quote_assets: USDT, BUSD, BTC, KRW
    origin_market = exchange_origin.split('/')[0]
    quote_asset_one = exchange_origin.split('/')[1]
    target_market = exchange_target.split('/')[0]
    quote_asset_two = exchange_target.split('/')[1]

    origin_market_df = self.exchange_websocket_dict[origin_market].get_price_df()
    origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
    target_market_df = self.exchange_websocket_dict[target_market].get_price_df()
    target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

    shared_bass_asset_list = list(set(origin_market_df['base_asset'].values).intersection(set(target_market_df['base_asset'].values)))
    origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
    target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)

    convert_rate = convert_asset_rate(origin_market, quote_asset_one, target_market, quote_asset_two)
    origin_market_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['tp','ap','bp']] * convert_rate
    # target_market_df[['converted_tp','converted_ap','converted_bp']] = target_market_df[['tp','ap','bp']]

    # divide by target_market_df[['tp','ap','bp']]
    premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
                            origin_market_df[['converted_tp','converted_bp','converted_ap']].values, columns=['tp_premium','LS_premium','SL_premium'])
    premium_df['LS_SL_spread'] = premium_df['LS_premium'] - premium_df['SL_premium']
    premium_df[['base_asset','quote_asset','tp','ap','bp','atp24h']] = target_market_df[['base_asset','quote_asset','tp','ap','bp','atp24h']]
    premium_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['converted_tp','converted_ap', 'converted_bp']]
    premium_df.loc[:, ['tp_premium','LS_premium','SL_premium','LS_SL_spread']] = premium_df[['tp_premium','LS_premium','SL_premium','LS_SL_spread']] * 100
    return premium_df

In [ ]:
# POSSIBLE quote_assets: USDT, BUSD, BTC, KRW

exchange_origin = 'BINANCE_USD_M/USDT'
exchange_target = 'UPBIT_SPOT/BTC'

origin_market = exchange_origin.split('/')[0]
quote_asset_one = exchange_origin.split('/')[1]
target_market = exchange_target.split('/')[0]
quote_asset_two = exchange_target.split('/')[1]


origin_market_df = core.exchange_websocket_dict[origin_market].get_price_df()
origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
target_market_df = core.exchange_websocket_dict[target_market].get_price_df()
target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

shared_bass_asset_list = list(set(origin_market_df['base_asset'].values).intersection(set(target_market_df['base_asset'].values)))
origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
# origin_market_df = origin_market_df[origin_market_df['base_asset'].isin(shared_bass_asset_list)].set_index('base_asset')
target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].sort_values('base_asset').reset_index(drop=True)
# target_market_df = target_market_df[target_market_df['base_asset'].isin(shared_bass_asset_list)].set_index('base_asset')

convert_rate = convert_asset_rate(origin_market, quote_asset_one, target_market, quote_asset_two)

origin_market_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['tp','ap','bp']] * convert_rate
# target_market_df[['converted_tp','converted_ap','converted_bp']] = target_market_df[['tp','ap','bp']]

# divide by target_market_df[['tp','ap','bp']]
premium_df = pd.DataFrame((target_market_df[['tp','ap','bp']].values - origin_market_df[['converted_tp','converted_bp','converted_ap']].values)/
                          origin_market_df[['converted_tp','converted_bp','converted_ap']].values, columns=['tp_premium','LS_premium','SL_premium'])
premium_df['LS_SL_spread'] = premium_df['LS_premium'] - premium_df['SL_premium']
premium_df[['base_asset','quote_asset','tp','ap','bp','atp24h']] = target_market_df[['base_asset','quote_asset','tp','ap','bp','atp24h']]
premium_df[['converted_tp','converted_ap','converted_bp']] = origin_market_df[['converted_tp','converted_ap', 'converted_bp']]
premium_df.loc[:, ['tp_premium','LS_premium','SL_premium','LS_SL_spread']] = premium_df[['tp_premium','LS_premium','SL_premium','LS_SL_spread']] * 100
premium_df.sort_values('atp24h', ascending=False).reset_index(drop=True).head(10)


In [ ]:
origin_market_df = core.exchange_websocket_dict[origin_market].get_price_df()
origin_market_df = origin_market_df[origin_market_df['quote_asset'] == quote_asset_one]
target_market_df = core.exchange_websocket_dict[target_market].get_price_df()
target_market_df = target_market_df[target_market_df['quote_asset'] == quote_asset_two]

In [ ]:
origin_market_df

In [ ]:
core.exchange_websocket_dict[target_market].get_price_df()

In [ ]:
target_market_df.head()[['base_asset','tp','ap','bp']]

In [ ]:
origin_market_df.head()[['base_asset','tp','ap','bp']]

In [ ]:
origin_market_df.head()

In [ ]:
convert_asset_rate(origin_market="BINANCE_SPOT", origin_quote_asset='BUSD', target_market="UPBIT_SPOT", target_quote_asset='KRW')

In [ ]:
core.update_dollar_return_dict['price']

In [ ]:
upbit_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['UPBIT'].upbit_ticker_dict)).T.reset_index()[['index','tp','scr','atp24h','h52wp','l52wp','ms','mw','tms']]
upbit_orderbook_df = pd.DataFrame(dict(core.exchange_websocket_dict['UPBIT'].upbit_orderbook_dict)).T.reset_index(drop=True)[['cd','tms','obu']]
upbit_orderbook_df['ap'] = upbit_orderbook_df['obu'].apply(lambda x: x[0]['ap'])
upbit_orderbook_df['bp'] = upbit_orderbook_df['obu'].apply(lambda x: x[0]['bp'])
upbit_orderbook_df.drop('obu', axis=1, inplace=True)
upbit_merged_df = pd.merge(upbit_ticker_df, upbit_orderbook_df, left_on='index', right_on='cd', how='inner')
upbit_merged_df['base_asset'] = upbit_merged_df['index'].apply(lambda x: x.split('-')[1])
upbit_merged_df['quote_asset'] = upbit_merged_df['index'].apply(lambda x: x.split('-')[0])
upbit_merged_df.drop('index', axis=1, inplace=True)
upbit_merged_df.loc[:, ['scr','atp24h','h52wp','l52wp','ap','bp']] = upbit_merged_df.loc[:, ['scr','atp24h','h52wp','l52wp','ap','bp']].astype(float)
upbit_merged_df.loc[:, 'scr'] = upbit_merged_df['scr'] * 100
upbit_merged_df.head()

In [ ]:
upbit_merged_df.head()

In [ ]:
core.info_dict['binance_spot_info_df'][['symbol','baseAsset','quoteAsset']].head()

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_SPOT'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_SPOT'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_spot_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df

In [ ]:
binance_merged_df['quote_asset'].unique()

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_USD_M'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_USD_M'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_usdm_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df

In [ ]:
binance_ticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_COIN_M'].ticker_dict)).T.reset_index(drop=True)[['s','P','c','v','q']]
binance_ticker_df.rename(columns={"q": "atp24h", 'P': 'scr', 'c': 'tp'}, inplace=True)
binance_bookticker_df = pd.DataFrame(dict(core.exchange_websocket_dict['BINANCE_COIN_M'].bookticker_dict)).T.reset_index(drop=True)[['s','b','a']]
binance_bookticker_df.rename(columns={"b": "bp", "a": "ap"}, inplace=True)
binance_merged_df = pd.merge(binance_ticker_df, binance_bookticker_df, on='s', how='inner')
binance_merged_df.loc[:, ['scr','tp','atp24h','ap','bp']] = binance_merged_df[['scr','tp','atp24h','ap','bp']].astype(float)
binance_merged_df = binance_merged_df.merge(core.info_dict['binance_coinm_info_df'][['symbol','baseAsset','quoteAsset']], left_on='s', right_on='symbol', how='inner')
binance_merged_df.drop(['symbol', 's'], axis=1, inplace=True)
binance_merged_df.rename(columns={'baseAsset':"base_asset", 'quoteAsset':"quote_asset"}, inplace=True)
binance_merged_df